# Py Lab 6 — Data visualization II
**PUBHLT 0411 · Python for Public Health Data Analysis**

Name: 

Submit this notebook as a `.ipynb` file.

---

Each exercise states the task and names the variables to create. Write your code in the empty cell below it and run the cell.

Use the exact variable names given — later exercises build on them.

## Before you start — the data

This lab reads:

- `pa_counties.csv`
- `pa_covid_monthly.csv`
- `pa_maternal_infant.csv`

These live in a `data/` folder next to this notebook. Run the cell below once to confirm Python can see them before you begin.

In [ ]:
# Setup — run this cell once, before anything else.
#
# The lab's CSVs live in a `data/` folder next to this notebook. This cell
# does not download anything; it just checks Python can find them.
import os

FILES = [
    "pa_counties.csv",
    "pa_covid_monthly.csv",
    "pa_maternal_infant.csv"
]

missing = [f for f in FILES if not os.path.exists(os.path.join("data", f))]

if missing:
    print("MISSING from data/:", ", ".join(missing))
    print("Working directory:", os.getcwd())
    print("Save this notebook into python/, next to data/, reopen it, and re-run this cell.")
else:
    print("Data ready:", ", ".join(FILES))

---

## Exercise 1 — Repair two seaborn calls

**Find and fix the problem in each figure.** The code below runs into an error, and even before the error, the first figure is wrong. Repair both calls in place.

- **Figure A** should give each of the six urbanicity categories its own colour, named in the legend.
- **Figure B** should draw every county in dark gold, `#8a6400`, with no legend.

**Create:**

- `births`: `data/pa_maternal_infant.csv` read as a DataFrame (67 rows, one per county).
- `counties`: `data/pa_counties.csv` read as a DataFrame (67 rows).
- `joined`: `births` merged with `counties` on `county` (67 rows, all columns of both).
- `fig_a`, `ax_a` and `fig_b`, `ax_b`: the figure and axes of each figure, from `plt.subplots()`.

**Draw:** Figure A, a scatter of `low_birthweight_pct` against `children_in_poverty_pct` whose legend names the six categories (not the numbers 1 to 6). Figure B, a scatter of `low_birthweight_pct` against `median_household_income` with every point dark gold and no legend.

In [ ]:
import pandas as pd                                        # tables
import seaborn as sns                                      # statistical figures
import matplotlib.pyplot as plt                            # figure and axes

births = pd.read_csv("data/pa_maternal_infant.csv")        # one row per county
counties = pd.read_csv("data/pa_counties.csv")             # urbanicity per county
joined = births.merge(counties, on="county")               # one table, 67 rows

# Figure A: one colour per urbanicity category
fig_a, ax_a = plt.subplots(figsize=(7, 3))                 # canvas and one axes
sns.scatterplot(data=joined, x="children_in_poverty_pct",  # data; poverty -> x
                y="low_birthweight_pct",                   # low birthweight -> y
                hue="urbanicity_code", ax=ax_a)            # MAPPING: category -> colour
ax_a.set(xlabel="Children in poverty (%)",                 # x-axis label with units
         ylabel="Low birthweight (%)")                     # y-axis label with units
plt.show()                                                 # render Figure A

# Figure B: every county in dark gold
fig_b, ax_b = plt.subplots(figsize=(7, 3))                 # canvas and one axes
sns.scatterplot(data=joined, x="median_household_income",  # data; income -> x
                y="low_birthweight_pct",                   # low birthweight -> y
                hue="#8a6400", ax=ax_b)                    # every point dark gold
ax_b.set(xlabel="Median household income ($)",             # x-axis label with units
         ylabel="Low birthweight (%)")                     # y-axis label with units
plt.show()                                                 # render Figure B

---

## Exercise 2 — Facets and position adjustments

Compare the distribution of **low birthweight** across metro status. Map `metro` to colour, make sure each histogram is fully visible, and facet **row-wise** by income.

**Create:**

- `births`, `counties`, `joined`: as in Exercise 1.
- `joined["income"]`: a new column, `"Higher income"` where `median_household_income` is above its statewide median ($62,382) and `"Lower income"` otherwise. `np.where()` does this in one line.
- `g`: the FacetGrid that `sns.displot()` returns.
- `medians`: a Series of the median `low_birthweight_pct` for each combination of `income` and `metro` (4 values), from `.groupby(["income", "metro"])`.

**Draw:** `sns.displot()` of `low_birthweight_pct` with `hue="metro"`, `row="income"` (two panels stacked in one column), `binwidth=0.5`, and no stacking: use `multiple="layer"` with `alpha=0.4`, or `element="step"`. Colour Metro `#003594` and Nonmetro `#8a6400`, title each panel with its income half, and label both axes.

**Data:**

```python
PAL = {"Metro": "#003594", "Nonmetro": "#8a6400"}          # course colours, by group
```

---

## Exercise 3 — Themes and scales

Draw a **boxplot** of low birthweight by metro status, styled as listed below.

**Create:**

- `births`, `counties`, `joined`: as in Exercise 1.
- `fig`, `ax`: the figure and its one axes.

**Draw:** `sns.boxplot()` with `metro` on x and `low_birthweight_pct` on y, and:

- the boxes coloured `#003594` (Metro) and `#8a6400` (Nonmetro), through `hue="metro"` and a `palette=` dictionary;
- a y-axis tick every **1 percentage point** (`MultipleLocator(1)` from `matplotlib.ticker`);
- the x-axis titled `"Metro status"`;
- a plain white background with no grid lines (`with sns.axes_style("white"):`);
- no legend, and a **bold** title (`fontweight="bold"`).

---

## Exercise 4 — Counts or rates

Compare COVID-19 deaths across Pennsylvania's six **regions** from August 2021 on, as a count and as a rate per 100,000 residents.

**Create:**

- `covid`: `data/pa_covid_monthly.csv`, with `month` converted to dates and only the rows from 2021-08-01 onward (2,144 rows).
- `counties`: as in Exercise 1.
- `deaths`: one row per county (67 rows), with `covid_deaths` summed over the months, merged with the `county`, `region`, and `population_2023` columns of `counties`.
- `by_region`: one row per region (6 rows, region as the index), with `covid_deaths` and `population_2023` summed, and `per_100k` (deaths ÷ population × 100,000).
- `by_count` and `by_rate`: `by_region` sorted ascending by `covid_deaths` and by `per_100k`.

**Draw:** two horizontal bar panels side by side with `ax.barh()`: counts from `by_count` on the left, rates from `by_rate` on the right, so the longest bar in each is at the top. Label each x-axis and title the panels `"Count"` and `"Rate"`.

---

## Exercise 5 — Layering: one group over the rest

Highlight the **North Central** counties over all counties, with **teen births** against **child poverty**.

**Create:**

- `births`, `counties`, `joined`: as in Exercise 1.
- `north_central`: the rows of `joined` whose `region` is `"North Central"` (14 rows).

**Draw:** two `sns.scatterplot()` calls on the same axes, with `children_in_poverty_pct` on x and `teen_births_per_1000` on y. First every county in `joined` in light grey, `#d1d5db`; then `north_central` in `#003594`, so it sits on top. Label both axes and say in the title which group is highlighted.

---

## Exercise 6 — Counting rows or averaging a column

Compare two statistics by **urbanicity**: how many counties fall in each category, and their mean low birthweight.

**Create:**

- `births`, `counties`, `joined`: as in Exercise 1.
- `order`: a list of the six `urbanicity` names from most to least urban, i.e. sorted by `urbanicity_code` from 1 to 6 (`"Large central metro"` first, `"Noncore"` last).
- `cnt`, `avg`: the left and right axes from `plt.subplots(1, 2, sharey=True)`.
- `means`: a Series of the mean `low_birthweight_pct` per category, in the order of `order` (6 values).

**Draw:** on `cnt`, `sns.countplot()` with `urbanicity` on y; on `avg`, `sns.barplot()` of `low_birthweight_pct` by `urbanicity` with `errorbar=None`. Pass `order=order` to both, and label each x-axis with what its bars measure.

---

## Exercise 7 — A diverging heatmap

Draw the correlations among all **seven** measures in the maternal file, showing only the cells **below the diagonal**. The upper triangle repeats them, and the diagonal is always 1.

**Create:**

- `births`: as in Exercise 1.
- `measures`: the seven column names of `births` other than `county`, in the file's order (`births.columns.drop("county")`).
- `r`: the correlation table `births[measures].corr()` (7 × 7).
- `mask`: a 7 × 7 boolean array, `True` on and above the diagonal: `np.triu(np.ones(r.shape, dtype=bool))`.
- `strongest_r`: a number, the correlation below the diagonal that is furthest from zero. `r.where(~mask).stack()` gives the 21 cells below the diagonal as a Series; `.abs().idxmax()` finds the pair.

**Draw:** `sns.heatmap()` of `r` with `mask=mask`, `annot=True`, `fmt=".2f"`, the diverging palette `cmap="BrBG"`, and the colour scale fixed with `vmin=-1, vmax=1`. Give the rows and columns short readable labels.

**Data:**

```python
short = ["Infant mort.", "Low bw", "Teen births", "Child mort.",  # readable labels, file order
         "Child pov.", "Uninsured", "Income"]             # ... matching measures
```

---

## Exercise 8 — Combine, save, and describe

Build a three-panel figure of **teen births**, save it to a file, and describe it in words.

**Create:**

- `births`, `counties`, `joined`: as in Exercise 1.
- `fig`, `axd`: from `plt.subplot_mosaic([["hist", "box"], ["scatter", "scatter"]])`. `axd` is a dictionary of three axes named `"hist"`, `"box"`, and `"scatter"`.
- `teen_births.png`: the figure saved with `fig.savefig(..., dpi=300)`, **before** `plt.show()`.
- `alt_text`: a string of one or two sentences saying what the figure shows and its main pattern, for a reader who cannot see it (at least 80 characters).

**Draw:** in `axd["hist"]`, a histogram of `teen_births_per_1000`; in `axd["box"]`, boxplots of it by `metro`; in `axd["scatter"]` (the full bottom row), teen births against `children_in_poverty_pct`, with `metro` mapped to both `hue=` and `style=`. Label every axis with units and give the figure one bold title with `fig.suptitle()`.

Use `PAL` from Exercise 2.